# Tutorial 2 — Transformer Internals

**Course:** Text & Language Processing / LLM Practical Track
**Format:** Hands-on notebook (Colab-ready)
**Suggested duration:** 120–150 minutes
**Prerequisite:** Tutorial 1 (Tokenization Deep Dive)

Tutorial 1 ended where the tokenizer does: a list of integer ids. This notebook follows those ids through
the model and back out as text.

By the end you will be able to:

- run a forward pass by hand and read the **shape** of everything it returns;
- explain what a **hidden state** is, and show that the same word has different ones in different contexts;
- read an **attention matrix**, and verify for yourself that a causal model cannot look forward;
- turn final hidden states into **logits**, then into a next-token distribution;
- write the **generation loop** yourself, and get the same output as `model.generate()`;
- choose a **decoding strategy** — greedy, temperature, top-k, top-p, beam search — and say what each
  one trades away;
- explain what the **KV cache** stores and measure what it saves;
- say where a model's **parameters** actually live.

## 0. Mental model: one forward pass

A forward pass is a fixed pipeline with no loops and no decisions:

![A forward pass drawn as tensors: input_ids of length 6 become a 6-by-d embedding matrix, pass through N transformer blocks that keep the same shape at every layer, produce 6 final hidden states, and are mapped by a shared LM head weight matrix to a 6-by-vocabulary logit matrix whose last row is used for generation](images/transformer.png)

Three things are worth fixing in your head before any code runs.

**Every stage keeps the sequence dimension.** A 6-token input produces 6 vectors at every layer and 6 rows
of logits at the end. The model scores the next token at *every* position simultaneously, not just the last
one — that is what makes training efficient. Generation simply ignores all rows but the last.

**Depth is refinement, not translation.** Layer 1 does not produce "words" and layer 12 "meaning". Each
block reads the whole sequence and writes an updated vector per position, gradually mixing context into
each one.

**One head, applied everywhere.** The LM head is a single weight matrix `W_vocab`, not one per position.
The same matrix maps every hidden state to vocabulary scores — which is why the logits come out as a
rectangle, and why a bigger vocabulary costs parameters (tutorial 1, section 4) rather than depth.

**Nothing here is generation.** One forward pass produces one distribution over next tokens. Generating
20 tokens means running this 20 times, which is where the KV cache in section 8 earns its keep.

## 1. Install the libraries

In [ ]:
!pip -q install -U transformers torch matplotlib

### Code walkthrough — installation

`torch` is the new one. Tutorials 0 and 1 used models only through `pipeline()` and tokenizers, which hid
the tensors; here we handle them directly, so PyTorch is no longer optional. `matplotlib` draws the
attention map in section 4.

## 2. Load a model and read its shape

`AutoModelForCausalLM` loads the weights *plus* the language-modelling head that turns hidden states into
vocabulary scores. `AutoModel` alone would give you the body without that head — useful for embeddings,
useless for generation.

We use GPT-2 (124M parameters) because it runs on a CPU in seconds and its architecture is the plain
decoder every modern LLM still resembles.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "gpt2"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, attn_implementation="eager")
model.eval()

cfg = model.config
print("layers:       ", cfg.n_layer)
print("attention heads per layer:", cfg.n_head)
print("hidden size:  ", cfg.n_embd)
print("vocab size:   ", cfg.vocab_size)
print("max context:  ", cfg.n_positions)
print()
total = sum(p.numel() for p in model.parameters())
print(f"total parameters: {total:,}")

### Code walkthrough — config and `eval()`

**`model.eval()`** switches the model out of training mode. For GPT-2 that disables dropout, so the same
input gives the same output every time. Forgetting it is a classic source of "why is my inference
non-deterministic?"

The config is the architecture, written down. **`n_layer`** is how many transformer blocks stack up,
**`n_head`** how many attention heads run in parallel inside each block, **`n_embd`** the width of the
vector carried at every position, and **`n_positions`** the longest sequence the model has position
embeddings for. Everything you print later is one of these numbers or a product of them.

Other architectures spell these differently — `num_hidden_layers`, `num_attention_heads`, `hidden_size`.
The concepts are identical; only the config keys move.

**`attn_implementation="eager"`** is there for one reason: section 4 needs the attention weights. Modern
Transformers defaults to a fused backend (SDPA / FlashAttention) that is faster but never materialises the
`seq × seq` matrix, so `output_attentions=True` comes back empty or falls back with a warning. The plain
"eager" path computes it explicitly. Keep it for inspection; drop it for production speed.


## 3. The forward pass

By default a model returns only logits. Asking for `output_hidden_states` and `output_attentions` makes it
hand back the intermediate values too, which is the whole point of this notebook.

In [ ]:
text = "The capital of France is"
inputs = tok(text, return_tensors="pt")

with torch.no_grad():
    out = model(**inputs, output_hidden_states=True, output_attentions=True)

n_tokens = inputs["input_ids"].shape[1]
print("tokens:", tok.convert_ids_to_tokens(inputs["input_ids"][0]))
print()
print("input_ids    ", tuple(inputs["input_ids"].shape), "  [batch, seq]")
print("logits       ", tuple(out.logits.shape), "  [batch, seq, vocab]")
print("hidden_states", len(out.hidden_states), "tensors of", tuple(out.hidden_states[0].shape), " [batch, seq, hidden]")
print("attentions   ", len(out.attentions), "tensors of", tuple(out.attentions[0].shape), "[batch, heads, seq, seq]")

### Code walkthrough — reading the shapes

**`torch.no_grad()`** tells PyTorch not to build a graph for backpropagation. We are not training, so the
gradient bookkeeping would be pure waste — roughly half the memory for nothing.

**`return_tensors="pt"`** you met in tutorial 1, section 8. It is why everything here is a tensor with a
batch dimension of 1 rather than a Python list.

Now the counts, which are the real content of this cell:

- **`hidden_states` has `n_layer + 1` entries**, not `n_layer`. Index `0` is the output of the *embedding*
  layer — the input before any block has touched it — and index `i` is the output of block `i`. The extra
  one is the "before" picture.
- **`attentions` has exactly `n_layer` entries**, one per block, each `[batch, heads, seq, seq]`. That last
  `seq × seq` is every position scoring every other position.
- **`logits` is `[batch, seq, vocab]`** — a full vocabulary-sized score vector at *every* position, as
  promised in section 0.

### Before you run

The next cell compares each token's vector before any block runs (`hidden_states[0]`) with its vector after
all twelve (`hidden_states[-1]`), using cosine similarity.

The similarities come back low — often near zero, sometimes negative. That is the point: the final vector at
a position is **not** a refined version of that word's embedding, it is a representation of that position
*in this sentence*. The embedding of `" France"` is the same tensor in every text that contains it; its
final hidden state is not.

This is what "contextual representation" means, and it is the single idea separating transformers from the
static word vectors (word2vec, GloVe) that preceded them.

In [ ]:
import torch.nn.functional as F

first = out.hidden_states[0][0]    # after embedding, before any block
last  = out.hidden_states[-1][0]   # after all 12 blocks

print(f"{'token':>10}  {'cos(embedding, final)':>22}  {'‖embedding‖':>12}  {'‖final‖':>10}")
for i, t in enumerate(tok.convert_ids_to_tokens(inputs["input_ids"][0])):
    sim = F.cosine_similarity(first[i], last[i], dim=0).item()
    print(f"{t:>10}  {sim:>22.3f}  {first[i].norm():>12.1f}  {last[i].norm():>10.1f}")

### Code walkthrough — how far the vectors travel

**`F.cosine_similarity(a, b, dim=0)`** measures the angle between two vectors, ignoring their lengths: `1.0`
means same direction, `0.0` unrelated, `-1.0` opposite.

The norms are printed alongside because they move too, and usually grow with depth. That is why cosine
similarity is the right comparison here — it isolates *direction* (roughly, "what this vector means") from
*magnitude*, which in transformers is entangled with layer normalisation and residual accumulation rather
than with meaning.

Try the same cell with a sentence where one word is ambiguous — `"The bank raised rates"` against
`"The river bank was steep"` — and compare the final hidden state of `bank` across the two. Same embedding
in, different representation out. That is the mechanism behind every downstream task in this course.

## 4. Attention: who looks at whom

Each attention head produces a `seq × seq` matrix. Row `i` is a probability distribution: how much position
`i` draws from every position when building its next representation.

Two properties are worth checking rather than taking on faith, and the next cell checks both.

In [ ]:
import numpy as np

LAYER, HEAD = 5, 4
attn = out.attentions[LAYER][0, HEAD].numpy()

print("shape:", attn.shape)
print("row sums (each row is a distribution):", attn.sum(axis=1).round(3))

upper = attn[np.triu_indices(n_tokens, k=1)]
print("largest weight above the diagonal:", f"{upper.max():.2e}")
print("→ a causal model cannot attend to the future" if upper.max() < 1e-6 else "→ unexpected: future leakage")

### Code walkthrough — two invariants

**Rows sum to 1.** Attention weights come out of a softmax over the key positions, so every row is a
probability distribution. A row is "where position `i` spent its attention budget", and the budget is
always exactly 1. If you ever see rows that do not sum to 1, you are looking at pre-softmax scores.

**Everything above the diagonal is zero.** `np.triu_indices(n, k=1)` selects the strictly upper triangle —
the positions *after* the current one. In a causal (decoder) model those are masked to `-inf` before the
softmax, so they come out as exactly 0. This is not a convention, it is the entire reason a decoder can be
trained on all positions at once: position 3 predicting token 4 must not have already seen token 4.

An encoder like BERT has no such mask, which is why BERT reads whole sentences bidirectionally but cannot
generate left to right.

In [ ]:
import matplotlib.pyplot as plt

labels = [t.replace("\u0120", " ").strip() or "␣" for t in tok.convert_ids_to_tokens(inputs["input_ids"][0])]

fig, ax = plt.subplots(figsize=(5.4, 4.6))
im = ax.imshow(attn, cmap="Blues", vmin=0.0, vmax=1.0)

ax.set_xticks(range(n_tokens), labels, rotation=45, ha="right")
ax.set_yticks(range(n_tokens), labels)
ax.set_xlabel("attended to  (key)")
ax.set_ylabel("attending from  (query)")
ax.set_title(f"Attention weights — layer {LAYER}, head {HEAD}", pad=10)

ax.tick_params(length=0)
for side in ax.spines.values():
    side.set_visible(False)

fig.colorbar(im, ax=ax, shrink=0.82, label="attention weight")
plt.tight_layout()
plt.show()

### Code walkthrough — reading the map

A single-hue ramp is deliberate. Attention weight is a **magnitude** running from 0 to 1, and magnitude
reads correctly only on one hue going light to dark. A rainbow map (`jet`, `rainbow`) would invent visual
boundaries where the data has none and collapses to mush for colourblind readers. **`vmin=0, vmax=1`** pins
the scale so that two heads plotted side by side are actually comparable — without it, matplotlib rescales
per plot and a weak head looks identical to a strong one.

Read it row by row, left to right. The lower-triangular shape is the causal mask you just verified. The
first column is usually dark across every row: with nothing else to attend to, heads dump leftover attention
budget on the first token. This is a well-documented behaviour known as an **attention sink**, and it is a
quirk of the softmax-must-sum-to-1 constraint rather than anything linguistic. It is not a minority effect:
in this model, 126 of the 144 heads send more than half their attention to position 0. Measure it yourself
by averaging `attentions[L][0, H][1:, 0]` across every layer and head.

Change `LAYER` and `HEAD` and re-run. Heads specialise: some track the previous token, some the first, some
attend to punctuation. Most look like noise, which is also worth knowing — interpretability is harder than
the tidy diagrams in papers suggest.

### Exercise 4.1

Plot the same head for the two "bank" sentences from section 3. Then find a head whose diagonal is offset by
one — a *previous-token head*, which attends almost entirely to position `i-1`. Try the early layers first.
What would such a head be useful for?

## 5. From hidden states to a next token

The **LM head** is a single linear layer mapping each `n_embd`-wide hidden state to one score per vocabulary
entry. Those raw scores are **logits**; a softmax turns them into probabilities.

In [ ]:
next_logits = out.logits[0, -1]              # last position only
probs = torch.softmax(next_logits, dim=-1)

print(f"prompt: {text!r}\n")
top = torch.topk(probs, 5)
for score, idx in zip(top.values, top.indices):
    print(f"{tok.decode(idx)!r:>12}  p = {score:.3f}")

print()
print("logits row:", tuple(next_logits.shape), "= one score per vocabulary entry")
print("probabilities sum to:", f"{probs.sum():.4f}")

### Code walkthrough — the last row only

**`out.logits[0, -1]`** takes batch item 0, last position. Every other row is discarded. Those rows are not
wasted during *training* — they supply a prediction target at every position, which is why a decoder learns
from an entire sequence in one pass — but at inference time only the final row can tell you what comes next.

**`torch.softmax`** exponentiates and normalises, turning arbitrary real-valued scores into a distribution
summing to 1. Note that softmax is *monotonic*: it never changes which token ranks highest. Greedy decoding
could read `argmax` straight off the logits. You need real probabilities the moment you want to sample, set
a temperature, or apply `top_p` — the subject of section 7.

`tok.decode(idx)` converts each id back to text, closing the loop from tutorial 1.

**A word about the output you just saw.** Tutorial 1's diagram used this same prompt and showed `" Paris"`
winning at 82%. GPT-2 small does not do that — its top token is `" the"`, at roughly 0.09, with the rest of
the distribution almost as flat. The diagram was illustrating the mechanism, not this checkpoint.

The gap is worth sitting with, because it is two lessons at once. A 124M-parameter model from 2019 is simply
not a reliable knowledge store. And a near-uniform top-5 is what *low confidence* looks like from the inside
— the same shape that sampling parameters act on in section 7. Re-run with a larger checkpoint,
say `Qwen/Qwen3-0.6B`, and watch the distribution sharpen.


## 6. Generation is this cell in a loop

Everything needed to generate text is now on the table: run a forward pass, take the last row, pick a token,
append it, repeat. The next cell does exactly that, then checks the result against `model.generate()`.

In [ ]:
ids = inputs["input_ids"]

for _ in range(8):
    with torch.no_grad():
        logits = model(ids).logits
    next_id = logits[0, -1].argmax().view(1, 1)
    ids = torch.cat([ids, next_id], dim=1)

manual = tok.decode(ids[0])

builtin = tok.decode(
    model.generate(**inputs, max_new_tokens=8, do_sample=False,
                   pad_token_id=tok.eos_token_id)[0]
)

print("by hand:  ", manual)
print("generate():", builtin)
print()
print("identical:", manual == builtin)

### Code walkthrough — the loop, and why it matches

**`.argmax()`** takes the highest-scoring token: greedy decoding. **`.view(1, 1)`** reshapes that scalar to
`[batch, seq]` so it can be concatenated, and **`torch.cat(..., dim=1)`** appends along the sequence axis.
The new ids become the next iteration's input — this is the "append it to the input" box from tutorial 1's
mental-model diagram.

The outputs match because **`do_sample=False`** with the default single beam *is* greedy decoding. If they
ever diverge for you, the usual cause is sampling being on.

**`pad_token_id=tok.eos_token_id`** silences a warning: GPT-2 ships without a pad token, and `generate()`
wants to know what to pad with even when, as here, nothing needs padding.

Worth noticing: this loop re-runs the model over the **entire** sequence every iteration. Step 8 recomputes
everything it already computed at step 7. That waste is exactly what section 8 removes.

## 7. Decoding: choosing from the distribution

Section 5 produced a probability distribution; section 6 took its `argmax`. That second step was a
**choice**, and it is a choice you make rather than one the model makes.

This matters more than it first appears. **Decoding is not part of the model.** The weights are fixed, the
forward pass is deterministic, and the distribution is whatever it is — everything below changes only how a
token is drawn from it. The same model produces careful prose or incoherent rambling depending on these
settings alone, which is why "the model is bad" is so often a decoding problem instead.

In [ ]:
common = dict(pad_token_id=tok.eos_token_id)

greedy_long = model.generate(**inputs, do_sample=False, max_new_tokens=40, **common)
print(tok.decode(greedy_long[0]))

### Code walkthrough — why greedy collapses

Greedy decoding takes the single highest-scoring token every step. It looks like the obvious choice — always
pick what the model thinks is most likely — and on open-ended text it reliably degenerates into a loop, as
it just did.

The mechanism is a feedback loop rather than a bug. Once `"the capital of the French Republic"` is in the
context, it becomes strong evidence for its own repetition, and because greedy is deterministic there is
nothing to break the cycle. Each repetition makes the next more likely.

Notice that the text is locally fluent throughout. Greedy is not producing *wrong* grammar; it is producing
the *highest-probability* sequence of words, and that turns out not to be the same thing as good text. The
most likely continuation of most sentences is bland, and the most likely continuation of a bland sentence is
the sentence again.

The deeper point: the model is not broken here. It assigned reasonable probabilities. The search over them
was what failed.

### 7.1 Temperature: reshaping before you sample

**Temperature** divides the logits before the softmax. Below 1 it sharpens the distribution toward the
leader; above 1 it flattens it toward uniform.

In [ ]:
with torch.no_grad():
    last_logits = model(**inputs).logits[0, -1]

print(f"prompt: {text!r}\n")
for T in (0.5, 1.0, 1.5):
    probs_T = torch.softmax(last_logits / T, dim=-1)
    top = torch.topk(probs_T, 4)
    row = "   ".join(f"{tok.decode(i).strip()!r}: {v:.3f}" for v, i in zip(top.values, top.indices))
    print(f"T = {T:<4}  {row}")

### Code walkthrough — what the numbers show

Watch the leading token across the three rows. At `T=0.5` it holds around 0.40; at `T=1.0` about 0.09; at
`T=1.5` roughly 0.02. The *ranking* never changes — division by a positive constant cannot reorder logits —
but the **gaps** do, and gaps are what sampling responds to.

So temperature does not make the model more or less knowledgeable. It adjusts how much the long tail of
unlikely tokens gets to participate:

- **`T < 1`** concentrates mass on the leaders. At the limit `T → 0` this *is* greedy decoding.
- **`T = 1`** leaves the model's own distribution untouched.
- **`T > 1`** lifts the tail, which buys variety and spends coherence.

Note how flat this distribution already is at `T=1` — the top token holds under 10%, and thousands of tokens
share the rest. That flatness is the model telling you it is unsure, and it is exactly the condition in which
temperature choices dominate the output. On a confident distribution, temperature barely matters.

### 7.2 Truncation: top-k and top-p

A flat distribution has a very long tail. Sampling from all 50,257 tokens means that any individual absurd
token is unlikely, but *some* absurd token is nearly certain over a few dozen steps.

Both fixes cut the tail before sampling. **top-k** keeps a fixed number of candidates; **top-p** (nucleus
sampling) keeps however many are needed to reach a cumulative probability, so the candidate set shrinks when
the model is confident and grows when it is not.

In [ ]:
from transformers import set_seed

strategies = {
    "greedy":                  dict(do_sample=False),
    "pure sampling (T=1)":     dict(do_sample=True, temperature=1.0, top_k=0, top_p=1.0),
    "top-k = 50":              dict(do_sample=True, temperature=1.0, top_k=50),
    "top-p = 0.9, T = 0.8":    dict(do_sample=True, temperature=0.8, top_p=0.9),
}

for name, settings in strategies.items():
    set_seed(0)
    sampled = model.generate(**inputs, max_new_tokens=25, **settings, **common)
    print(f"--- {name}")
    print(tok.decode(sampled[0]), "\n")

### Code walkthrough — reading the four outputs

**`set_seed(0)`** before each call makes sampling reproducible. Without it these cells give different text
every run, which makes comparing strategies impossible. Note that it does nothing for greedy, which has no
randomness to seed.

**`top_k=0, top_p=1.0`** disables both filters — that row is *pure* sampling from the full distribution, and
it usually reads as fluent nonsense, wandering off into unrelated vocabulary. This is the long tail doing
exactly what the previous cell predicted.

**`top_k=50`** keeps the 50 most likely tokens and renormalises. Simple and effective, but the fixed cutoff is
crude: 50 candidates is far too many when the model is certain and sometimes too few when it is genuinely
torn.

**`top_p=0.9`** keeps the smallest set of tokens whose probabilities sum to 0.9. On a confident distribution
that might be two tokens; on the flat one above, dozens. Adapting to the model's own confidence is why
nucleus sampling became the default for open-ended generation.

**The last row is the practical default** — `top_p` around 0.9 with temperature slightly below 1. Judge it
honestly though: the output is fluent, grammatical, and quite possibly false. Decoding fixed the
*degeneration* while doing nothing at all for *accuracy*. No sampling parameter can make a 124M model know
things.

### 7.3 Beam search

Greedy commits to the best token at each step. **Beam search** keeps `num_beams` partial sequences alive and
chooses the one with the best total probability at the end — a better search for the high-probability
sequence.

In [ ]:
beam = model.generate(**inputs, num_beams=4, do_sample=False,
                      early_stopping=True, max_new_tokens=25, **common)
print(tok.decode(beam[0]))

### Code walkthrough — a better search for the wrong thing

Beam search does find a higher-probability sequence than greedy. It also loops, often just as badly — which
is the clearest possible evidence for the point made at the top of this section: repetition is not a
search failure. The
high-probability region of the space genuinely *is* repetitive, so searching it harder finds more repetition.

That gives a clean rule:

- **Use beam search for closed-ended tasks** — translation, summarisation, anything with a roughly correct
  answer worth searching for. `num_beams=4` or `5` is standard.
- **Do not use it for open-ended generation.** You want text that is interesting, not text that is likely,
  and those objectives diverge.

Beams cost linearly: `num_beams=4` is about four times the compute and memory of greedy.

**`early_stopping=True`** halts once all beams have finished rather than exhausting `max_new_tokens`.

### 7.4 Repetition controls

If a loop persists, two arguments attack it directly.

In [ ]:
fixed = model.generate(**inputs, do_sample=False, max_new_tokens=40,
                       repetition_penalty=1.2, no_repeat_ngram_size=3, **common)
print(tok.decode(fixed[0]))

### Code walkthrough — two blunt instruments

**`repetition_penalty=1.2`** divides the logits of tokens already present in the context, making them less
likely each time they recur. `1.0` disables it; `1.1`–`1.3` is the usable band. Push it higher and the model
starts avoiding words it *needs* — articles, prepositions, the subject of the sentence — because it has no
notion of which repetition is legitimate.

**`no_repeat_ngram_size=3`** forbids any 3-token sequence from appearing twice. It is absolute, which makes it
effective and dangerous: a name or technical term that genuinely must repeat now *cannot*. Be wary of it in
factual or code generation.

Compare this output with the greedy run at the top of this section. The loop is gone, and the same caveat applies — it is no
more accurate than before, only more varied. These are cosmetic fixes to a symptom, and the better fix for
repetition in a serious application is usually a larger or instruction-tuned model rather than more
aggressive penalties.

### 7.5 When generation stops

Three mechanisms, and a model that "won't stop" is nearly always missing one.

In [ ]:
unbounded = model.generate(**inputs, do_sample=False, max_new_tokens=40, **common)
stopped   = model.generate(**inputs, do_sample=False, max_new_tokens=40,
                           stop_strings=["."], tokenizer=tok, **common)

print("max_new_tokens only:\n", tok.decode(unbounded[0]), "\n")
print("stop_strings=['.']:\n", tok.decode(stopped[0]), "\n")
print("eos token:", repr(tok.eos_token), "| id:", tok.eos_token_id)

### Code walkthrough — the three stopping mechanisms

**`max_new_tokens`** is the hard ceiling, and the only one guaranteed to fire. Always set it. It is your
protection against an unbounded bill.

**The EOS token** ends generation when the model emits it. Base models like GPT-2 rarely do — nothing in
plain web text marks "the document ends here" — which is why this run needs a length cap. Instruction-tuned
models emit EOS reliably, because that behaviour is precisely what instruction tuning teaches. If a chat
model never stops, suspect a prompt-format problem: an incorrectly applied chat template leaves the model
unsure it is even its turn.

**`stop_strings=["."]`** halts at a text pattern, and needs **`tokenizer=tok`** passed alongside so the
generator can decode while it goes. This is how you stop at a delimiter you defined — `"\n\n"`, a closing
brace, or a role marker. The output above is visibly cut at the first full stop.

A subtlety worth knowing: stop strings are *trimmed after the fact*, not predicted. The tokens are generated,
then cut. You pay for them either way.

### 7.6 Streaming

Generation is sequential, so tokens exist one at a time. Waiting for all of them before showing anything is a
choice, and usually the wrong one for an interface.

In [ ]:
from transformers import TextStreamer

_ = model.generate(**inputs, do_sample=False, max_new_tokens=20,
                   streamer=TextStreamer(tok, skip_prompt=True), **common)

### Code walkthrough — streaming changes nothing but the wait

**`TextStreamer(tok, skip_prompt=True)`** prints each token as it is produced; `skip_prompt=True` suppresses
the echoed input. The return value is discarded here because the text has already been printed.

Total time is unchanged — this is purely about **perceived** latency. Time-to-first-token becomes the number
the user feels, rather than time-to-last-token, and the two can differ by many seconds on a long answer.

For an application rather than a terminal, use **`TextIteratorStreamer`**, which yields strings to your own
code instead of printing, and runs `generate` on a background thread so a web handler can forward tokens as
they arrive. Every chat interface you have used works this way.

### Exercise 7.1

Take one prompt and generate it eight times with `top_p=0.9, temperature=0.8`, **without** calling
`set_seed`. How different are the eight? Now do the same with `temperature=0.3`, then `temperature=1.5`.

Then pick the setting you would ship for (a) a creative writing assistant, (b) a SQL generator, and (c) a
customer-service reply drafter — and write one sentence each on why. There is no single right answer, which
is the point: decoding parameters are a product decision, not a technical default.

## 8. The KV cache

Attention at position `i` needs a key and a value vector for every position `≤ i`. In the loop above, those
were recomputed from scratch on every iteration — for a 500-token generation, position 1's key gets computed
500 times.

The **KV cache** keeps them. Each step then computes keys and values for the *one* new token and reads the
rest from memory, turning generation from quadratic in total work to linear.

In [ ]:
import time

def time_generation(use_cache, new_tokens=60):
    t0 = time.perf_counter()
    model.generate(**inputs, max_new_tokens=new_tokens, do_sample=False,
                   use_cache=use_cache, pad_token_id=tok.eos_token_id)
    return time.perf_counter() - t0

with_cache = time_generation(True)
without    = time_generation(False)

print(f"use_cache=True :  {with_cache:.2f}s")
print(f"use_cache=False:  {without:.2f}s")
print(f"speed-up:         {without / with_cache:.1f}×")

### Code walkthrough — what you just measured

Both calls produce **identical text**. The cache is a pure efficiency optimisation; it changes nothing about
what the model computes, only how often it recomputes it.

The speed-up grows with sequence length, because the work saved is quadratic while the work kept is linear.
At 60 tokens on a CPU the gap is modest; at 2,000 tokens it is the difference between a usable product and
an unusable one. Run it again with `new_tokens=200` to watch the ratio move.

The cost is memory, and it is not small: roughly `2 × layers × heads × head_dim × seq_len × batch ×
bytes_per_value`. That is why long-context serving is a memory problem before it is a compute problem, and
why techniques aimed at shrinking this cache — multi-query and grouped-query attention, paged attention —
matter so much in production systems.

`use_cache=True` is the default. You will essentially never turn it off outside a demonstration like this
one.

## 9. Where the parameters are

"124M parameters" is not one undifferentiated blob. Knowing its split explains several practical things at
once — why vocabulary size is expensive, and why fine-tuning strategies target the blocks.

In [ ]:
emb_tokens = model.transformer.wte.weight.numel()
emb_pos    = model.transformer.wpe.weight.numel()
embeddings = emb_tokens + emb_pos
blocks     = total - embeddings

print(f"token embeddings     {emb_tokens:>12,}   ({emb_tokens/total:.1%})")
print(f"position embeddings  {emb_pos:>12,}   ({emb_pos/total:.1%})")
print(f"transformer blocks   {blocks:>12,}   ({blocks/total:.1%})")
print(f"{'':21}{'-'*12}")
print(f"total                {total:>12,}")
print()
print(f"per block            {blocks // cfg.n_layer:>12,}")
print(f"weights in fp32      {total * 4 / 1e6:>12.0f} MB")

### Code walkthrough — reading the budget

**`wte`** is the token embedding table, `vocab_size × n_embd` — one learned vector per vocabulary entry.
**`wpe`** is the position table, `n_positions × n_embd`. Both are plain lookup tables, not computation.

The token embeddings are a striking share of a small model. This is the direct cost of tutorial 1's
vocabulary-size decision: every extra token in the vocabulary buys a whole row of `n_embd` parameters, and
for a 124M model that table alone is around a third of the total. In a 7B model the same table is a rounding
error — which is why large models can afford far bigger vocabularies, and why they tokenize non-English text
more efficiently.

**`total * 4`** bytes is fp32 storage, 4 bytes per parameter. Halve it for fp16/bf16, quarter it for int8.
That arithmetic is the whole of "will this model fit on my GPU?" — and section 8's KV cache is the term
people forget to add.

## 10. Three things that will bite you

**`eval()` and `no_grad()` are different switches.** `eval()` changes layer *behaviour* (dropout, batchnorm);
`no_grad()` stops gradient *tracking*. Inference wants both. Neither is on by default.

**Padding without a mask corrupts results.** Batch two sentences of different lengths and the short one is
padded — then attention happily attends to the pad tokens. `tokenizer(...)` returns an `attention_mask`
precisely so this does not happen, and passing `**inputs` forwards it. Build `input_ids` by hand and you
must pass the mask by hand too.

**A tokenizer/model mismatch stays silent here as well.** Tutorial 1 made this point for decoding; it holds
for embeddings. Id 2000 indexes row 2000 of `wte` whatever tokenizer produced it, so the wrong pairing gives
you fluent nonsense rather than an error.

### Exercise 10.1

Tokenize two sentences of different lengths together with `padding=True`, run a forward pass, and compare the
final hidden state of the *last real token* of the short sentence with the one you get from that sentence
alone. They should match. Then re-run while passing only `input_ids` and dropping `attention_mask`, and watch
them stop matching.

That difference is a padding bug, and it is entirely silent — no error, no warning, just quietly worse
numbers.

## Glossary — quick reference

| Term | Meaning |
|---|---|
| **Forward pass** | One run of input through the network: ids in, logits out. No loop, no decisions. |
| **Hidden state** | The vector held at one position after a given layer; `[batch, seq, hidden]`. |
| **Contextual representation** | A hidden state that depends on the whole sequence, not just its own token. |
| **Attention matrix** | Per head, a `seq × seq` table whose rows are distributions over positions attended to. |
| **Causal mask** | Zeroing the upper triangle so a position cannot attend to later ones. |
| **Attention sink** | Heads parking surplus attention on the first token, because rows must sum to 1. |
| **LM head** | The linear layer mapping a hidden state to one score per vocabulary entry. |
| **Logits** | Raw pre-softmax scores. Monotonic with probability, so `argmax` is unaffected by softmax. |
| **Decoding** | How a token is drawn from the distribution. Not part of the model; entirely your choice. |
| **Greedy decoding** | Always take the `argmax`. Deterministic, and degenerates into loops on open-ended text. |
| **Temperature** | Divides logits before softmax. `<1` sharpens, `>1` flattens; never reorders. |
| **top-k** | Sample from the `k` most likely tokens. A fixed cutoff. |
| **top-p (nucleus)** | Sample from the smallest set reaching cumulative probability `p`. Adapts to confidence. |
| **Beam search** | Keeps several partial sequences and picks the best overall. For closed-ended tasks only. |
| **`repetition_penalty`** | Down-weights tokens already in the context. Usable band ~1.1–1.3. |
| **`no_repeat_ngram_size`** | Hard ban on repeating any n-gram. Effective, and dangerous for terms that must recur. |
| **EOS token** | The token that ends generation. Base models rarely emit it; instruction-tuned models do. |
| **Streaming** | Emitting tokens as produced. Changes perceived latency only, never total time. |
| **KV cache** | Stored keys/values for past positions, making generation linear instead of quadratic. |
| **`eval()` vs `no_grad()`** | Layer behaviour vs gradient tracking — two independent switches, both wanted at inference. |

---